



## Cellule 1 — Imports & Ingestion des données

### Objectif
Charger les 5 fichiers CSV de façon professionnelle : gestion de l'encodage, optimisation des `dtypes`, et affichage des **shapes** à chaque étape pour tracer les éventuelles pertes.

---

### Stratégie d'Architecture & Jointures

Notre pipeline repose sur **trois couches fonctionnelles** :

| Couche | Fichier(s) | Rôle | Clé de jointure |
|---|---|---|---|
| **Interactions (CF)** | `collaborative_books_df.csv` | Table centrale — notes réelles + mappings entiers | `book_id_mapping`, `user_id_mapping` |
| **Métadonnées (CB)** | `collaborative_book_metadata.csv` | Texte riche pour TF-IDF (desc, genre, auteur) | `book_id_mapping` → JOIN LEFT sur la couche CF |
| **Décodage identités** | `user_id_map.csv`, `book_id_map.csv`, `book_titles.csv` | Rétablir UUIDs / titres en sortie finale | `user_id_csv`, `book_id_csv` → `book_id` |

**Pourquoi cette stratégie ?**
- Les colonnes `*_mapping` sont déjà des **entiers continus 0-based** → zero-cost pour construire les matrices `scipy.sparse` sans re-encodage.
- On ne joint **jamais** les tables de décodage dans la table de travail : elles sont gardées séparées et appelées uniquement à l'affichage.
- `book_titles.csv` (1.4M lignes) est converti directement en `dict` puis supprimé → aucun DataFrame inutile en mémoire.

---

### Lecture rapide
- présence des fichiers
- colonnes clés bien chargées
- taux de recouvrement entre interactions et métadonnées

In [ ]:
# =============================================================================
# CELLULE 1 — Imports + Chargement simple des fichiers CSV
# =============================================================================

# 1) Imports
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy import sparse
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

print(" Librairies importées")
print(f"   NumPy: {np.__version__} | Pandas: {pd.__version__}")

# 2) Chemins des fichiers
DATA_DIR = Path("Dataset")
PATHS = {
    "interactions": DATA_DIR / "collaborative_books_df.csv",
    "metadata": DATA_DIR / "collaborative_book_metadata.csv",
    "book_id_map": DATA_DIR / "book_id_map.csv",
    "user_id_map": DATA_DIR / "user_id_map.csv",
    "book_titles": DATA_DIR / "book_titles.csv",
}

# Vérifier que tous les fichiers existent
missing_files = [name for name, path in PATHS.items() if not path.exists()]
if missing_files:
    print(f" Fichiers manquants: {missing_files}")
else:
    print(" Tous les fichiers sont présents\n")

# 3) Charger la table principale: interactions
# Cette table contient les notes réelles utilisateur-livre.
interaction_cols = ["user_id_mapping", "book_id_mapping", "book_id", "title", "Actual Rating"]
interaction_dtypes = {
    "user_id_mapping": "int32",
    "book_id_mapping": "int32",
    "book_id": "int32",
    "title": "category",
    "Actual Rating": "int8",
}

print("── Chargement: collaborative_books_df.csv")
interactions_raw = pd.read_csv(
    PATHS["interactions"],
    usecols=interaction_cols,
    dtype=interaction_dtypes,
    encoding="utf-8",
)

loaded_interaction_cols = set(interactions_raw.columns)
expected_interaction_cols = set(interaction_cols)
missing_interaction_cols = sorted(expected_interaction_cols - loaded_interaction_cols)
if missing_interaction_cols:
    print(f" Colonnes manquantes dans interactions: {missing_interaction_cols}")
else:
    print(" Colonnes interactions OK")

print(f"Shape interactions: {interactions_raw.shape}")
print(
    f"Plage user_id_mapping: [{interactions_raw['user_id_mapping'].min()} - "
    f"{interactions_raw['user_id_mapping'].max()}]"
)
print(
    f"Plage book_id_mapping: [{interactions_raw['book_id_mapping'].min()} - "
    f"{interactions_raw['book_id_mapping'].max()}]"
)
print(
    f"Plage Actual Rating: [{interactions_raw['Actual Rating'].min()} - "
    f"{interactions_raw['Actual Rating'].max()}]"
)
print()

# 4) Charger les métadonnées livres (utile pour le Content-Based)
metadata_cols = [
    "book_id_mapping", "book_id", "title", "description", "genre",
    "name", "num_pages", "ratings_count", "image_url", "url"
]
metadata_dtypes = {
    "book_id_mapping": "int32",
    "book_id": "int32",
    "num_pages": "float32",
    "ratings_count": "int32",
}

print("── Chargement: collaborative_book_metadata.csv")
metadata_raw = pd.read_csv(
    PATHS["metadata"],
    usecols=metadata_cols,
    dtype=metadata_dtypes,
    encoding="utf-8",
)

if "book_id_mapping" not in metadata_raw.columns:
    print(" Clé 'book_id_mapping' absente des métadonnées")
else:
    print(" Clé de jointure metadata OK")
print(f"Shape metadata: {metadata_raw.shape}\n")

# 5) Charger les tables de mapping (utiles pour décodage / affichage final)
print("── Chargement: user_id_map.csv")
user_id_map = pd.read_csv(
    PATHS["user_id_map"],
    dtype={"user_id_csv": "int32", "user_id": "object"},
    encoding="utf-8",
)
print(f"Shape user_id_map: {user_id_map.shape}")

print("── Chargement: book_id_map.csv")
book_id_map = pd.read_csv(
    PATHS["book_id_map"],
    dtype={"book_id_csv": "int32", "book_id": "int32"},
    encoding="utf-8",
)
print(f"Shape book_id_map: {book_id_map.shape}")

print("── Chargement: book_titles.csv -> dict")
titles_tmp = pd.read_csv(
    PATHS["book_titles"],
    usecols=["title", "book_id"],
    dtype={"book_id": "int32"},
    encoding="utf-8",
)
BOOK_TITLE_LOOKUP = dict(zip(titles_tmp["book_id"], titles_tmp["title"]))
del titles_tmp
print(f"Entrées dans BOOK_TITLE_LOOKUP: {len(BOOK_TITLE_LOOKUP):,}\n")

# 6) Vérifier le recouvrement interactions <-> metadata
# Question: combien de livres de la table interactions ont aussi des métadonnées ?
interaction_books = set(interactions_raw["book_id_mapping"].unique())
metadata_books = set(metadata_raw["book_id_mapping"].unique())
overlap_keys = interaction_books & metadata_books

n_books_interactions = len(interaction_books)
n_books_metadata = len(metadata_books)
coverage_pct = (len(overlap_keys) / n_books_interactions) * 100 if n_books_interactions else 0.0

print("── Recouvrement interactions ↔ metadata")
print(f"Livres uniques interactions: {n_books_interactions}")
print(f"Livres uniques metadata:     {n_books_metadata}")
print(f"Intersection:                {len(overlap_keys)} ({coverage_pct:.1f}% couverts)")
print()

# 7) Petit résumé final
print("=" * 55)
print("RÉCAPITULATIF — ÉTAT INITIAL DES DONNÉES")
print("=" * 55)
summary_data = {
    "Table": ["interactions", "metadata", "user_id_map", "book_id_map"],
    "Lignes": [len(interactions_raw), len(metadata_raw), len(user_id_map), len(book_id_map)],
    "Colonnes": [
        interactions_raw.shape[1],
        metadata_raw.shape[1],
        user_id_map.shape[1],
        book_id_map.shape[1],
    ],
}
print(pd.DataFrame(summary_data).to_string(index=False))
print("=" * 55)

---

## Cellule 2 — Fusion & Architecture des tables de travail

### Objectif
Construire les **deux DataFrames de travail** qui alimenteront l'ensemble du pipeline :

| DataFrame | Contenu | Usage |
|---|---|---|
| `df_interactions` | Interactions nettoyées (user × book × note) | Filtrage de sparsité, matrice CF, MF/SVD |
| `df_content` | Métadonnées enrichies (une ligne par livre) | Corpus TF-IDF, Content-Based |

### Stratégie de jointure
- **LEFT JOIN** `interactions_raw` ← `metadata_raw` sur `book_id_mapping`  
  → On conserve **toutes** les interactions, même si un livre n'a pas de métadonnées (NULL pour le CB uniquement).  
  → On trace la perte : lignes avant vs après, nombre de livres sans métadonnées.
- `df_content` = `metadata_raw` dédupliqué sur `book_id_mapping` (une ligne = un livre unique).

### Vérification rapide
- comparaison simple des lignes avant/après JOIN
- comptage des livres avec et sans métadonnées
- aperçu final de `df_interactions` et `df_content`

In [ ]:
# =============================================================================
# CELLULE 2 — Fusion simple des tables (version facile)
# =============================================================================

# 1) Enlever les doublons metadata (1 livre = 1 ligne)
# Si un livre apparait plusieurs fois, on garde la première ligne.
meta_before = len(metadata_raw)
metadata_dedup = metadata_raw.drop_duplicates(subset="book_id_mapping", keep="first")
meta_after = len(metadata_dedup)

print("── Étape 1: Déduplication metadata")
print(f"Avant: {meta_before:,} | Après: {meta_after:,} | Doublons retirés: {meta_before - meta_after:,}")
print()

# 2) Fusion LEFT JOIN : on garde TOUTES les interactions
# Même si un livre n'a pas de metadata, la ligne interaction reste présente.
meta_cols_for_join = ["book_id_mapping", "description", "genre", "name", "num_pages", "ratings_count", "image_url", "url"]
rows_before_join = len(interactions_raw)

df_interactions = interactions_raw.merge(
    metadata_dedup[meta_cols_for_join],
    on="book_id_mapping",
    how="left",
)
rows_after_join = len(df_interactions)

print("── Étape 2: LEFT JOIN interactions + metadata")
print(f"Lignes avant JOIN: {rows_before_join:,}")
print(f"Lignes après JOIN: {rows_after_join:,}")
if rows_after_join == rows_before_join:
    print(" Verification rapide: le LEFT JOIN a gardé le même nombre de lignes")
else:
    print(" Verification rapide: le nombre de lignes a changé après le JOIN")
print()

# 3) Mesurer la couverture metadata dans df_interactions
# Un livre sans metadata a souvent description = NaN.
books_total = df_interactions["book_id_mapping"].nunique()
books_no_meta = df_interactions[df_interactions["description"].isna()]["book_id_mapping"].nunique()
books_with_meta = books_total - books_no_meta
coverage = (books_with_meta / books_total) * 100 if books_total else 0.0

print("── Étape 3: Couverture metadata")
print(f"Livres total: {books_total:,}")
print(f"Livres avec metadata: {books_with_meta:,} ({coverage:.1f}%)")
print(f"Livres sans metadata: {books_no_meta:,} ({100 - coverage:.1f}%)")
print()

# 4) Renommer les colonnes pour la suite du notebook
# Plus simple: rating + author

df_interactions = df_interactions.rename(columns={
    "Actual Rating": "rating",
    "name": "author",
})

print("── Étape 4: Colonnes finales df_interactions")
print(df_interactions.columns.tolist())
print(f"Shape df_interactions: {df_interactions.shape}")
print()

# 5) Construire df_content (1 ligne par livre)
# Table dédiée au Content-Based: texte + infos livre.
df_content = metadata_dedup.copy()

# Si le titre metadata est vide, on récupère le titre depuis interactions
title_from_interactions = (
    interactions_raw[["book_id_mapping", "title"]]
    .drop_duplicates(subset="book_id_mapping")
    .set_index("book_id_mapping")["title"]
    .astype(str)
)

df_content["title"] = df_content["title"].fillna(df_content["book_id_mapping"].map(title_from_interactions))
df_content = df_content.rename(columns={"name": "author"})

print("── Étape 5: df_content prêt")
print(f"Shape df_content: {df_content.shape}")
print(df_content.columns.tolist())
print()

# 6) Résumé final de la cellule
print("=" * 60)
print("RÉSUMÉ CELLULE 2")
print("=" * 60)
summary = {
    "Table": ["df_interactions", "df_content"],
    "Lignes": [len(df_interactions), len(df_content)],
    "Colonnes": [df_interactions.shape[1], df_content.shape[1]],
    "Usage": ["CF / MF / évaluation", "Content-Based (TF-IDF)"],
}
print(pd.DataFrame(summary).to_string(index=False))
print("=" * 60)

---

## Cellule 3 — Nettoyage des données 

### Objectif
Nettoyer les 2 tables de travail pour éviter les erreurs dans les modèles.

### Table 1 : `df_interactions`
On corrige 3 problèmes :
- NaN dans les colonnes clés (`user_id_mapping`, `book_id_mapping`, `rating`)
- notes hors plage [1, 5]
- doublons `(user_id_mapping, book_id_mapping)`

### Table 2 : `df_content`
On corrige les colonnes texte et numériques :
- `title` manquant -> suppression de la ligne
- `author` manquant -> "unknown"
- `description` -> nettoyage texte (HTML, espaces)
- `genre` -> normalisation texte
- `num_pages` invalide -> remplacement par la médiane

### Vérification rapide
- résumé des lignes supprimées
- compte simple des NaN restants
- shape finale des 2 tables

In [ ]:
# =============================================================================
# CELLULE 3 — Nettoyage des données (version simple)
# =============================================================================
import re

# ----------------------------------------------------------------------------
# A) Nettoyage de df_interactions
# ----------------------------------------------------------------------------
print("=" * 60)
print("A) Nettoyage de df_interactions")
print("=" * 60)

shape_0 = df_interactions.shape
print(f"Shape initiale: {shape_0}")

# A1) Supprimer les lignes avec NaN dans les colonnes clés
key_cols = ["user_id_mapping", "book_id_mapping", "rating"]
nan_key_mask = df_interactions[key_cols].isna().any(axis=1)
n_nan_keys = int(nan_key_mask.sum())
df_interactions = df_interactions.loc[~nan_key_mask].copy()
print(f"A1 - Lignes supprimées (NaN colonnes clés): {n_nan_keys}")
print(f"     Shape: {df_interactions.shape}")

# A2) Supprimer les notes hors [1, 5]
invalid_rating_mask = ~df_interactions["rating"].between(1, 5)
n_invalid = int(invalid_rating_mask.sum())
df_interactions = df_interactions.loc[~invalid_rating_mask].copy()
print(f"A2 - Lignes supprimées (notes invalides): {n_invalid}")
print(f"     Shape: {df_interactions.shape}")

# A3) Supprimer les doublons (user, book) en gardant la dernière note
n_before_dedup = len(df_interactions)
df_interactions = df_interactions.drop_duplicates(
    subset=["user_id_mapping", "book_id_mapping"],
    keep="last",
)
n_removed_dup = n_before_dedup - len(df_interactions)
print(f"A3 - Doublons supprimés (user, book): {n_removed_dup}")
print(f"     Shape: {df_interactions.shape}")

# A4) Verification rapide final interactions
remaining_nan_keys = int(df_interactions[key_cols].isna().sum().sum())
remaining_invalid = int((~df_interactions["rating"].between(1, 5)).sum())
remaining_dups = int(df_interactions.duplicated(subset=["user_id_mapping", "book_id_mapping"]).sum())
print(
    "A4 - Verification rapide interactions: "
    f"NaN clés={remaining_nan_keys}, "
    f"ratings hors plage={remaining_invalid}, "
    f"doublons={remaining_dups}"
)

print(" df_interactions nettoyé")
print(f"   Lignes supprimées au total: {shape_0[0] - len(df_interactions)}")

# ----------------------------------------------------------------------------
# B) Nettoyage de df_content
# ----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("B) Nettoyage de df_content")
print("=" * 60)

content_cols = ["title", "author", "description", "genre", "num_pages"]
print("NaN avant nettoyage:")
print(df_content[content_cols].isna().sum().to_string())

shape_b0 = df_content.shape
print(f"\nShape initiale: {shape_b0}")

# B1) Supprimer livres sans titre
n_no_title = int(df_content["title"].isna().sum())
df_content = df_content.dropna(subset=["title"]).copy()
print(f"B1 - Lignes supprimées (title manquant): {n_no_title}")
print(f"     Shape: {df_content.shape}")

# B2) Auteur manquant -> 'unknown'
df_content["author"] = df_content["author"].fillna("unknown").astype(str).str.strip()
print("B2 - author nettoyé (NaN -> 'unknown')")

# B3) Nettoyage description
# On garde les lignes même si description vide (utile pour coverage global).
def clean_text(text) -> str:
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)   # retire balises HTML
    text = re.sub(r"&\w+;", " ", text)     # retire entités HTML
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_content["description"] = df_content["description"].apply(clean_text)
n_empty_desc = int((df_content["description"] == "").sum())
print(f"B3 - descriptions vides après nettoyage: {n_empty_desc}")

# B4) Nettoyage genre
def clean_genre(genre) -> str:
    if pd.isna(genre) or str(genre).strip() in ("", "[]", "nan"):
        return ""
    g = str(genre)
    g = re.sub(r"[\[\]\"'`]", " ", g)
    g = re.sub(r"[,;]", " ", g)
    g = re.sub(r"\s+", " ", g).strip().lower()
    return g

df_content["genre"] = df_content["genre"].apply(clean_genre)
n_empty_genre = int((df_content["genre"] == "").sum())
print(f"B4 - genres vides après nettoyage: {n_empty_genre}")

# B5) Nettoyage num_pages
valid_pages = df_content.loc[df_content["num_pages"] > 0, "num_pages"]
median_pages = valid_pages.median()
n_bad_pages = int(((df_content["num_pages"].isna()) | (df_content["num_pages"] <= 0)).sum())
df_content["num_pages"] = (
    df_content["num_pages"]
    .where(df_content["num_pages"] > 0, other=np.nan)
    .fillna(median_pages)
)
print(f"B5 - num_pages corrigé: {n_bad_pages} valeurs remplacées par la médiane ({median_pages:.0f})")

# B6) Verification rapide final content
remaining_title_nan = int(df_content["title"].isna().sum())
remaining_author_nan = int(df_content["author"].isna().sum())
remaining_pages_nan = int(df_content["num_pages"].isna().sum())
print(
    "B6 - Verification rapide content: "
    f"title NaN={remaining_title_nan}, "
    f"author NaN={remaining_author_nan}, "
    f"num_pages NaN={remaining_pages_nan}"
)

print("\nNaN après nettoyage:")
nan_after = df_content[content_cols].isna().sum()
print(nan_after.to_string())

print(f"\n df_content nettoyé - shape finale: {df_content.shape}")
print(f"   Lignes supprimées au total: {shape_b0[0] - len(df_content)}")

# ----------------------------------------------------------------------------
# Résumé final
# ----------------------------------------------------------------------------
print("\n" + "=" * 60)
print("RÉCAPITULATIF — APRÈS NETTOYAGE")
print("=" * 60)
recap_clean = {
    "DataFrame": ["df_interactions", "df_content"],
    "Shape finale": [str(df_interactions.shape), str(df_content.shape)],
}
print(pd.DataFrame(recap_clean).to_string(index=False))
print("=" * 60)

---

## Cellule 4 — EDA 

### Objectif
Faire une exploration visuelle rapide de `df_interactions` pour répondre à 4 questions simples :
1. Les notes sont-elles équilibrées ou trop positives ?
2. Quels livres reçoivent le plus de notes ?
3. Combien d'utilisateurs et de livres sont en situation de cold start ?
4. La matrice user × book est-elle très creuse (sparse) ?

### Ce que cette cellule va produire
- Figure 1 : distribution des notes + Top 15 livres les plus notés
- Figure 2 : histogrammes cold start (users et books, échelle log)
- Figure 3 : heatmap d'un échantillon de la matrice
- Un résumé chiffré final (users, books, interactions, sparsité)

### Définitions utiles
- **Sparsité** = $1 - \frac{\text{interactions}}{\text{users} \times \text{books}}$
- **Cold start user** : utilisateur avec moins de 5 notes
- **Cold start book** : livre avec moins de 3 notes

In [ ]:
# =============================================================================
# CELLULE 4 — EDA 
# =============================================================================

# 1) Métriques de base
ratings_per_user = df_interactions.groupby("user_id_mapping")["rating"].count()
ratings_per_book = df_interactions.groupby("book_id_mapping")["rating"].count()

n_users = df_interactions["user_id_mapping"].nunique()
n_books = df_interactions["book_id_mapping"].nunique()
n_interactions = len(df_interactions)
sparsity = 1 - n_interactions / (n_users * n_books)

print("── Métriques globales")
print(f"Utilisateurs uniques : {n_users:,}")
print(f"Livres uniques       : {n_books:,}")
print(f"Interactions totales : {n_interactions:,}")
print(f"Sparsité             : {sparsity*100:.4f}%")
print(f"Densité              : {(1 - sparsity)*100:.4f}%")
print()

# 2) Figure 1 : Distribution des notes + Top 15 livres
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("EDA — Vue globale des notes", fontsize=14, fontweight="bold")

# 2.1 Distribution des notes
rating_counts = df_interactions["rating"].value_counts().sort_index()
axes[0].bar(
    rating_counts.index.astype(str),
    rating_counts.values,
    color=sns.color_palette("muted", 5),
    edgecolor="white",
    linewidth=0.8,
 )
axes[0].set_title("4.1 — Distribution des notes")
axes[0].set_xlabel("Note")
axes[0].set_ylabel("Nombre d'interactions")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

for bar, val in zip(axes[0].patches, rating_counts.values):
    pct = val / n_interactions * 100
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + n_interactions * 0.005,
        f"{pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9,
    )

# 2.2 Top 15 livres les plus notés
top15_books = ratings_per_book.nlargest(15).reset_index()
top15_books.columns = ["book_id_mapping", "n_ratings"]

id_to_title = (
    df_interactions[["book_id_mapping", "title"]]
    .drop_duplicates("book_id_mapping")
    .set_index("book_id_mapping")["title"]
    .astype(str)
)

def shorten_title(t, max_len=30):
    if pd.isna(t):
        return "Unknown title"
    t = str(t)
    return t if len(t) <= max_len else t[:max_len] + "…"

top15_books["title_short"] = top15_books["book_id_mapping"].map(id_to_title).apply(shorten_title)

axes[1].barh(
    top15_books["title_short"],
    top15_books["n_ratings"],
    color=sns.color_palette("muted")[0],
    edgecolor="white",
)
axes[1].invert_yaxis()
axes[1].set_title("4.2 — Top 15 livres les plus notés")
axes[1].set_xlabel("Nombre de notes reçues")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.tight_layout()
plt.savefig("eda_fig1_ratings_top15.png", dpi=120, bbox_inches="tight")
plt.show()
print(" Figure 1 sauvegardée : eda_fig1_ratings_top15.png\n")

# 3) Figure 2 : Cold Start (users/books)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("EDA — Analyse du Cold Start", fontsize=14, fontweight="bold")

# 3.1 Notes par utilisateur
axes[0].hist(
    ratings_per_user.values,
    bins=50,
    color=sns.color_palette("muted")[1],
    edgecolor="white",
    log=True,
)
cold_start_users = (ratings_per_user < 5).sum()
axes[0].axvline(5, color="crimson", linestyle="--", linewidth=1.5, label="Seuil = 5")
axes[0].set_title("4.3 — Notes par utilisateur (log)")
axes[0].set_xlabel("Nombre de notes données")
axes[0].set_ylabel("Nb d'utilisateurs (log)")
axes[0].legend()
axes[0].text(
    0.97,
    0.95,
    f"Cold start users: {cold_start_users:,}\n({cold_start_users/n_users*100:.1f}% < 5 notes)",
    transform=axes[0].transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8),
)

# 3.2 Notes par livre
axes[1].hist(
    ratings_per_book.values,
    bins=50,
    color=sns.color_palette("muted")[2],
    edgecolor="white",
    log=True,
)
cold_start_books = (ratings_per_book < 3).sum()
axes[1].axvline(3, color="crimson", linestyle="--", linewidth=1.5, label="Seuil = 3")
axes[1].set_title("4.4 — Notes par livre (log)")
axes[1].set_xlabel("Nombre de notes reçues")
axes[1].set_ylabel("Nb de livres (log)")
axes[1].legend()
axes[1].text(
    0.97,
    0.95,
    f"Cold start books: {cold_start_books:,}\n({cold_start_books/n_books*100:.1f}% < 3 notes)",
    transform=axes[1].transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8),
)

plt.tight_layout()
plt.savefig("eda_fig2_cold_start.png", dpi=120, bbox_inches="tight")
plt.show()
print(" Figure 2 sauvegardée : eda_fig2_cold_start.png\n")

# 4) Figure 3 : Heatmap densité (échantillon 50x50 pour lisibilité)
top50_users = ratings_per_user.nlargest(50).index
top50_books = ratings_per_book.nlargest(50).index

sample_matrix = (
    df_interactions[
        df_interactions["user_id_mapping"].isin(top50_users)
        & df_interactions["book_id_mapping"].isin(top50_books)
    ]
    .pivot_table(
        index="user_id_mapping",
        columns="book_id_mapping",
        values="rating",
        aggfunc="mean",
    )
    .reindex(index=top50_users, columns=top50_books)
)

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    sample_matrix,
    ax=ax,
    cmap="YlOrRd",
    mask=sample_matrix.isna(),
    linewidths=0.3,
    linecolor="lightgrey",
    cbar_kws={"label": "Note moyenne"},
    xticklabels=False,
    yticklabels=False,
)

density_sample = sample_matrix.notna().sum().sum() / (50 * 50)
ax.set_title(
    "4.5 — Densité matrice user × book\n"
    f"(échantillon: {density_sample*100:.1f}% | global: {sparsity*100:.2f}% )",
    fontsize=11,
)
ax.set_xlabel("Livres")
ax.set_ylabel("Utilisateurs")

plt.tight_layout()
plt.savefig("eda_fig3_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()
print(" Figure 3 sauvegardée : eda_fig3_heatmap.png\n")

# 5) Résumé final
print("=" * 60)
print("RÉCAPITULATIF EDA")
print("=" * 60)
print(f"Utilisateurs         : {n_users:,}")
print(f"Livres               : {n_books:,}")
print(f"Interactions         : {n_interactions:,}")
print(f"Sparsité globale     : {sparsity*100:.4f}%")
print(f"Note moyenne         : {df_interactions['rating'].mean():.3f} / 5")
print(f"Note médiane         : {df_interactions['rating'].median():.1f} / 5")
print(f"Cold start users     : {cold_start_users:,} ({cold_start_users/n_users*100:.1f}%)")
print(f"Cold start books     : {cold_start_books:,} ({cold_start_books/n_books*100:.1f}%)")
print("=" * 60)

print(df_interactions['rating'].value_counts().sort_index())

# 6) Diagnostic explicite du biais de notation
low_ratings = int(rating_counts.get(1, 0) + rating_counts.get(2, 0))
neutral_ratings = int(rating_counts.get(3, 0))
high_ratings = int(rating_counts.get(4, 0) + rating_counts.get(5, 0))

low_pct = (low_ratings / n_interactions) * 100 if n_interactions else 0.0
neutral_pct = (neutral_ratings / n_interactions) * 100 if n_interactions else 0.0
high_pct = (high_ratings / n_interactions) * 100 if n_interactions else 0.0

imbalance_ratio = (high_ratings / low_ratings) if low_ratings > 0 else np.inf

print("\n" + "-" * 60)
print("DIAGNOSTIC BIAIS DE NOTATION")
print("-" * 60)
print(f"Ratings bas (1-2)     : {low_ratings:,} ({low_pct:.2f}%)")
print(f"Ratings neutres (3)   : {neutral_ratings:,} ({neutral_pct:.2f}%)")
print(f"Ratings hauts (4-5)   : {high_ratings:,} ({high_pct:.2f}%)")
print(f"Ratio hauts/bas       : {imbalance_ratio:.2f}x")

if high_pct >= 60:
    print(" Dataset biaisé vers les notes élevées (positive bias).")
else:
    print(" Biais modéré des notes.")

title_consistency = df_interactions.groupby('book_id_mapping')['title'].nunique()
print("Livres avec plusieurs titres :", (title_consistency > 1).sum())

In [ ]:
# =============================================================================
# CELLULE 4.1 — Poids de classes (anti-biais de notes)
# =============================================================================

if "df_interactions" not in globals():
    raise ValueError("df_interactions introuvable. Exécute d'abord les cellules de préparation.")

# Groupes de ratings
# low: 1-2 | mid: 3 | high: 4-5
rating_series = df_interactions["rating"]

group_labels = np.select(
    [rating_series <= 2, rating_series == 3, rating_series >= 4],
    ["low_1_2", "mid_3", "high_4_5"],
    default="unknown",
)

group_counts = pd.Series(group_labels).value_counts()

total = int(group_counts.sum())
n_groups = 3

# Poids inverses à la fréquence: w_g = total / (n_groups * count_g)
group_weights = {
    g: (total / (n_groups * int(group_counts[g])))
    for g in ["low_1_2", "mid_3", "high_4_5"]
    if g in group_counts and group_counts[g] > 0
}

# Poids par rating individuel (1..5)
rating_to_group = {1: "low_1_2", 2: "low_1_2", 3: "mid_3", 4: "high_4_5", 5: "high_4_5"}
rating_class_weights = {
    r: float(group_weights[rating_to_group[r]])
    for r in [1, 2, 3, 4, 5]
    if rating_to_group[r] in group_weights
}

# Exemple: vecteur sample_weight aligné sur df_interactions
sample_weight = rating_series.map(rating_class_weights).astype("float32")

print("=" * 64)
print("POIDS DE CLASSES — ANTI-BIAIS DE NOTES")
print("=" * 64)
print("Comptes par groupe:")
print(group_counts.reindex(["low_1_2", "mid_3", "high_4_5"]).fillna(0).astype(int).to_string())
print("\nPoids par groupe:")
print(pd.Series(group_weights).reindex(["low_1_2", "mid_3", "high_4_5"]).to_string())
print("\nPoids par rating:")
print(pd.Series(rating_class_weights).sort_index().to_string())
print(f"\nSample weight ready: shape={sample_weight.shape}, dtype={sample_weight.dtype}")
print("=" * 64)

# Utilisation type:
# - sklearn metrics: mean_squared_error(y_true, y_pred, sample_weight=sample_weight)
# - entraînement (modèles qui acceptent sample_weight): fit(X, y, sample_weight=sample_weight)

## **Audit de qualité des données (avant filtrage)**

### Validation systématique
- **Missing values** : vérifier les valeurs NULL
- **Duplicates** : détecter les doublons exacts
- **Mappings** : validation des encodages ID
- **Rating range** : vérifier la validité des notes
- **Sparsity** : calculer le taux de complétude
- **Activity filtering** : profils d'activité utilisateur/livre
- **Metadata coverage** : complétude des colonnes descriptives
- **Consistency checks** : cohérence inter-tables

In [ ]:
# =============================================================================
# CELLULE 4.5 — Audit de qualité des données (avant filtrage)
# =============================================================================

print("="*70)
print("AUDIT DE QUALITÉ — AVANT FILTRAGE")
print("="*70)
print()

# 1) Missing Values
print("1⃣  MISSING VALUES")
print("─" * 70)
missing_counts = df_interactions.isnull().sum()
missing_pct = (df_interactions.isnull().sum() / len(df_interactions) * 100)
missing_df = pd.DataFrame({
    "Colonne": missing_counts.index,
    "Count NULL": missing_counts.values,
    "% NULL": missing_pct.values
})
missing_df = missing_df[missing_df["Count NULL"] > 0] if (missing_df["Count NULL"] > 0).any() else missing_df.head(0)
print(missing_df.to_string(index=False) if len(missing_df) > 0 else " Aucune valeur manquante détectée")
print()

# 2) Duplicates
print("2⃣  DUPLICATES")
print("─" * 70)
exact_dupes = df_interactions.duplicated().sum()
user_book_dupes = df_interactions.duplicated(subset=['user_id_mapping', 'book_id_mapping']).sum()
print(f"Doublons exacts (toutes colonnes)   : {exact_dupes:,}")
print(f"Doublons (user, book) pairs         : {user_book_dupes:,}")
if user_book_dupes > 0:
    print("  Attention: Le même utilisateur a noté le même livre plusieurs fois")
print()

# 3) Mappings validity
print("3⃣  MAPPINGS VALIDITY")
print("─" * 70)
user_mapping_valid = (df_interactions['user_id_mapping'] >= 0).all()
book_mapping_valid = (df_interactions['book_id_mapping'] >= 0).all()
print(f"user_id_mapping valides (≥0)       : {user_mapping_valid}")
print(f"book_id_mapping valides (≥0)       : {book_mapping_valid}")
print(f"User ID range                      : [{df_interactions['user_id_mapping'].min()} - {df_interactions['user_id_mapping'].max()}]")
print(f"Book ID range                      : [{df_interactions['book_id_mapping'].min()} - {df_interactions['book_id_mapping'].max()}]")
print()

# 4) Rating range & distribution
print("4⃣  RATING RANGE & DISTRIBUTION")
print("─" * 70)
print(f"Min rating                         : {df_interactions['rating'].min()}")
print(f"Max rating                         : {df_interactions['rating'].max()}")
print(f"Mean rating                        : {df_interactions['rating'].mean():.2f}")
print(f"Median rating                      : {df_interactions['rating'].median():.2f}")
print(f"Std dev                            : {df_interactions['rating'].std():.2f}")
print()
print("Répartition des notes:")
rating_dist = df_interactions['rating'].value_counts().sort_index()
for rating, count in rating_dist.items():
    pct = count / len(df_interactions) * 100
    print(f"  Rating {rating:g} : {count:6,} ({pct:5.2f}%)")
print()

# 5) Sparsity (avant filtrage)
print("5⃣  SPARSITY (avant filtrage)")
print("─" * 70)
n_users = df_interactions['user_id_mapping'].nunique()
n_books = df_interactions['book_id_mapping'].nunique()
n_interactions = len(df_interactions)
max_possible = n_users * n_books
sparsity_pct = (1 - n_interactions / max_possible) * 100
density_pct = (n_interactions / max_possible) * 100
print(f"Users (uniques)                    : {n_users:,}")
print(f"Books (uniques)                    : {n_books:,}")
print(f"Interactions (total)               : {n_interactions:,}")
print(f"Max possible interactions          : {max_possible:,}")
print(f"Density                            : {density_pct:.4f}%")
print(f"Sparsity                           : {sparsity_pct:.2f}%")
print()

# 6) Activity filtering (distribution d'activité)
print("6⃣  ACTIVITY FILTERING")
print("─" * 70)
user_activity = df_interactions.groupby('user_id_mapping')['rating'].count()
book_activity = df_interactions.groupby('book_id_mapping')['rating'].count()

print("Activité utilisateurs (notes par user):")
print(f"  Min                              : {user_activity.min()}")
print(f"  Max                              : {user_activity.max()}")
print(f"  Mean                             : {user_activity.mean():.1f}")
print(f"  Median                           : {user_activity.median():.1f}")
print(f"  Q1 (25%)                         : {user_activity.quantile(0.25):.1f}")
print(f"  Q3 (75%)                         : {user_activity.quantile(0.75):.1f}")
print()
print("Activité livres (notes par book):")
print(f"  Min                              : {book_activity.min()}")
print(f"  Max                              : {book_activity.max()}")
print(f"  Mean                             : {book_activity.mean():.1f}")
print(f"  Median                           : {book_activity.median():.1f}")
print(f"  Q1 (25%)                         : {book_activity.quantile(0.25):.1f}")
print(f"  Q3 (75%)                         : {book_activity.quantile(0.75):.1f}")
print()

# 7) Metadata coverage
print("7⃣  METADATA COVERAGE")
print("─" * 70)
metadata_cols = ['title', 'authors', 'author', 'description', 'genre', 'image_url']
for col in metadata_cols:
    if col in df_interactions.columns:
        non_null = df_interactions[col].notna().sum()
        coverage = non_null / len(df_interactions) * 100
        print(f"{col:20} : {non_null:6,} / {len(df_interactions):,} ({coverage:5.2f}%)")
    elif col in df_content.columns:
        non_null = df_content[col].notna().sum()
        coverage = non_null / len(df_content) * 100
        print(f"{col:20} : {non_null:6,} / {len(df_content):,} ({coverage:5.2f}%)")
print()

# 8) Consistency checks
print("8⃣  CONSISTENCY CHECKS")
print("─ " * 35)

# 8a) Title consistency
title_consistency = df_interactions.groupby('book_id_mapping')['title'].nunique()
inconsistent_books = (title_consistency > 1).sum()
print(f"Livres avec plusieurs titres       : {inconsistent_books}")
if inconsistent_books > 0:
    print(f"    {inconsistent_books} books avoir discrepancies — utiliser le titre modal")
print()

# 8b) Diagnostic: Check what columns exist where
print("DIAGNOSTIC — Colonnes disponibles:")
print("─" * 70)
print(f"df_interactions.columns ({len(df_interactions.columns)}): {df_interactions.columns.tolist()}")
print(f"\ndf_content.columns ({len(df_content.columns)}): {df_content.columns.tolist()}")
print()

# 8c) User-Book overlap — vérifier les colonnes disponibles
print("Tentative de fusion df_interactions ← df_content...")

# Identify common metadata columns (excluding join key and rating)
common_metadata = set(df_interactions.columns) & set(df_content.columns)
common_metadata.discard('book_id_mapping')
common_metadata.discard('rating')

print(f"Colonnes communes (metadata): {list(common_metadata)}")

if common_metadata:
    # Merge - les colonnes communes vont être dupliquées, pandas ajoute des suffixes
    df_merged = df_interactions.merge(
        df_content[['book_id_mapping'] + list(common_metadata)],
        on='book_id_mapping', how='left',
        suffixes=('_interactions', '_content')
    )

    print(f"Après merge, colonnes: {df_merged.columns.tolist()}")

    # Compte les interactions avec metadata complète
    metadata_cols_content = [col for col in df_merged.columns if col.endswith('_content')]
    if metadata_cols_content:
        orphan_count = df_merged[metadata_cols_content].isnull().all(axis=1).sum()
        covered_books = len(df_merged) - orphan_count
        print(f"\nInteractions avec metadata : {covered_books:,} / {len(df_merged):,}")
        print(f"  → Metadata columns: {metadata_cols_content}")
    else:
        print("  PROBLÈME: Les colonnes metadata ont disparu après la fusion")
else:
    print("  Pas de colonnes metadata communes entre df_interactions et df_content")
    print("   Cela signifie que df_interactions a déjà tout ce qu'on a besoin")
    print("   Pas besoin de faire un merge!")

    # Vérifier si les données sont déjà là
    metadata_in_interactions = [
        col for col in ['title', 'authors', 'author', 'description', 'genre']
        if col in df_interactions.columns
    ]
    if metadata_in_interactions:
        print(f"    Ces colonnes sont déjà dans df_interactions: {metadata_in_interactions}")
        print()
        print(" CONCLUSION: df_interactions contient DÉJÀ toutes les métadonnées!")
        print("   Aucun merge additionnel n'est nécessaire.")
        print("   Les données sont prêtes pour la modélisation.")
    else:
        print("  Avertissement: Pas de metadata détectée du tout!")

print()
print("="*70)


## **Détection des livres dupliqués **

### Problème
Même livre peut apparaître avec **plusieurs IDs** à cause de:
- Variations dans les titres (accents, espaces, troncatures)
- Différentes éditions ("Hardcover", "Paperback", "eBook")
- Erreurs de saisie dans les données sources

Impact: **recommandations fragmentées** pour le même livre

In [ ]:
import pandas as pd
from difflib import SequenceMatcher

# ==========================================
# 4.6: AUDIT DES DOUBLONS DE LIVRES
# ==========================================
print("\n" + "=" * 70)
print("4.6 - AUDIT DES DOUBLONS DE LIVRES (Exact + Semantic)")
print("=" * 70)

# Initialize defaults for all variables
exact_dupes = 0
potential_dupes_df = pd.DataFrame()
total_books = 0
unique_titles = 0
title_author_unique = 0

# 1) Doublons exacts sur (title, authors/author/name)
print("1⃣  DOUBLONS EXACTS (même titre + même auteur)")
print("─" * 70)

# Find the author column (might be 'authors', 'author', or 'name')
author_col = None
for col in ['authors', 'author', 'name']:
    if col in df_content.columns:
        author_col = col
        break

if author_col and 'title' in df_content.columns:
    book_meta = df_content[['book_id_mapping', 'title', author_col]].copy()
    book_meta.rename(columns={author_col: 'author'}, inplace=True)
    exact_dupes = book_meta.duplicated(subset=['title', 'author'], keep=False).sum()
    print(f"Livres avec titre ET auteur ({author_col}) identiques : {exact_dupes}")
    
    if exact_dupes > 0:
        dup_groups = book_meta[book_meta.duplicated(subset=['title', 'author'], keep=False)].sort_values('title')
        print("\nExemples de doublons exacts:")
        for title in dup_groups['title'].unique()[:5]:
            ids = dup_groups[dup_groups['title'] == title]['book_id_mapping'].tolist()
            print(f"  {title[:60]} → IDs: {ids}")
else:
    print(f"Colonnes disponibles dans df_content: {df_content.columns.tolist()}")
    print("  Impossible de faire la comparaison: 'title' ou colonne auteur manquante")
    book_meta = None
print()

# 2) Doublons potentiels (titre très similaire)
print("2⃣  DOUBLONS POTENTIELS (titres similaires > 85%)")
print("─" * 70)

def string_similarity(a, b):
    """Calcule similarité entre 2 strings (0-1)"""
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

potential_dupes = []
if book_meta is not None:
    book_titles = list(book_meta['title'].dropna().unique())

    # Vérifier les titres similaires (> 85% match)
    for i, title1 in enumerate(book_titles):
        for title2 in book_titles[i+1:]:
            sim = string_similarity(title1, title2)
            if sim > 0.85 and sim < 1.0:  # 85-99% de similarité = suspect
                potential_dupes.append({
                    'title1': title1,
                    'title2': title2,
                    'similarity': sim
                })

    potential_dupes_df = pd.DataFrame(potential_dupes).sort_values('similarity', ascending=False) if potential_dupes else pd.DataFrame()

    print(f"Pairs de titres similaires (85%-99%) : {len(potential_dupes_df)}")
    if len(potential_dupes_df) > 0:
        print("\nTop 15 doublons potentiels:")
        for idx, row in potential_dupes_df.head(15).iterrows():
            ids1 = book_meta[book_meta['title'] == row['title1']]['book_id_mapping'].tolist()
            ids2 = book_meta[book_meta['title'] == row['title2']]['book_id_mapping'].tolist()
            print(f"  {row['similarity']:.1%} | {row['title1'][:40]:40s} vs {row['title2'][:40]:40s}")
            print(f"         → IDs {ids1} vs {ids2}")
    else:
        print(" Pas de titres similaires détectés")
else:
    print("  Impossible de calculer: données manquantes")
print()

# 3) Analyse des doublons exacts par nombre de notes
print("3⃣  IMPACT DES DOUBLONS — Fragmentation des notes")
print("─" * 70)

if book_meta is not None and author_col and exact_dupes > 0:
    # Pour chaque groupe de livres dupliqués, calculer l'impact
    dup_titles = book_meta[book_meta.duplicated(subset=['title', 'author'], keep=False)]['title'].unique()
    
    fragmentation_impact = []
    for title in dup_titles[:10]:  # Top 10 pour l'affichage
        book_ids = book_meta[book_meta['title'] == title]['book_id_mapping'].tolist()
        
        # Compter les notes pour chaque ID
        rating_counts = []
        total_ratings = 0
        for bid in book_ids:
            count = (df_interactions['book_id_mapping'] == bid).sum()
            rating_counts.append(count)
            total_ratings += count
        
        if total_ratings > 0:
            fragmentation_impact.append({
                'title': title,
                'num_ids': len(book_ids),
                'book_ids': book_ids,
                'rating_counts': rating_counts,
                'total_ratings': total_ratings
            })
    
    if fragmentation_impact:
        print(f"\nFragmentation détectée (notes dispersées):")
        for item in sorted(fragmentation_impact, key=lambda x: x['total_ratings'], reverse=True):
            print(f"  Titre: {item['title'][:60]}")
            for bid, cnt in zip(item['book_ids'], item['rating_counts']):
                print(f"    → ID {bid}: {cnt:,} notes")
            print(f"      TOTAL: {item['total_ratings']:,} notes sur {item['num_ids']} IDs (fragmentation)")
            print()
else:
    print(" Pas de doublons exacts détectés → pas de fragmentation")

print()

# 4) Résumé et recommandations
print("4⃣  RÉSUMÉ & RECOMMANDATIONS")
print("─" * 70)

if book_meta is not None:
    total_books = book_meta['book_id_mapping'].nunique()
    unique_titles = book_meta['title'].nunique()
    title_author_unique = book_meta.drop_duplicates(subset=['title', 'author']).shape[0]

    print(f"Total books (IDs)                      : {total_books:,}")
    print(f"Unique titles                          : {unique_titles:,}")
    print(f"Unique (title, author) pairs           : {title_author_unique:,}")
    print(f"Exact duplicates (title+author)        : {exact_dupes:,}")
    print(f"Potential semantic duplicates          : {len(potential_dupes_df):,}")
    print()

    print("RECOMMANDATIONS:")
    if exact_dupes > 0:
        print("  ACTION REQUIRED: Doublons exacts détectés!")
        print("   Solution: Fusionner par (title, author) → consolider les ratings")
        print("   Impact: Pourrait augmenter le poids de ces livres populaires")
    else:
        print(" Pas de doublons exacts → Système OK")

    if len(potential_dupes_df) > 0:
        print(f"\n  À VÉRIFIER: {len(potential_dupes_df)} paires de titres similaires")
        print("   Actions: Revue manuelle + fuzzy matching + Levenshtein distance")
    else:
        print("\n Pas de titres similaires → Fondation solide")
        
    print("\nConclusion: Système de doublons = ", end="")
    if exact_dupes > 0 or len(potential_dupes_df) > 5:
        print("  À améliorer (voir actions ci-dessus)")
    else:
        print(" ROBUSTE")
else:
    print("  Impossible de générer le résumé: données manquantes")
    print("Vérifiez que 'title' et une colonne auteur existent dans df_content")

print()